<div class="alert alert-danger" role="alert">
<h1 align="center"><font size = 14>Burgers (PINN)</font></h1>
<h4 align="center">Aug, 15_2026<h4>
<h3 align="center">Armin Amani</h3>

<div class="alert alert-danger" role="alert">
📤 Import Libraries

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pyDOE import lhs
import time

<div class="alert alert-danger" role="alert"> 
🔎 Set device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

<div class="alert alert-danger" role="alert"> 
🔎 seeds for reproducibility

In [ ]:
seed = 1234
torch.manual_seed(seed)
np.random.seed(seed)

<div class="alert alert-danger" role="alert"> 
🔎 Problem parameters

In [ ]:
nu = 0.01 / np.pi  # Kinematic viscosity
L = 1.0  # Spatial domain half-length
T = 1.0  # Time domain

<div class="alert alert-danger" role="alert"> 
🔎 Domain boundaries

In [ ]:
x_min, x_max = -L, L
t_min, t_max = 0.0, T
ub = np.array([t_max, x_max])
lb = np.array([t_min, x_min])

<div class="alert alert-danger" role="alert"> 
🔎 Number of boundary, initial, and collocation points

In [ ]:
N_i = 1000  # Initial condition points
N_b = 1000  # Boundary condition points (per boundary)
N_c = 10000  # Collocation points

def getData():
    # Initial condition at t=0
    t_i = np.zeros((N_i, 1))
    x_i = lb[1] + (ub[1] - lb[1]) * lhs(1, N_i)
    tx_i = np.concatenate([t_i, x_i], axis=1)
    u_i = -np.sin(np.pi * x_i)

    # Boundary conditions at x=-1 and x=1
    t_b = lb[0] + (ub[0] - lb[0]) * lhs(1, N_b)
    x_b1 = np.full((N_b, 1), -L)
    x_b2 = np.full((N_b, 1), L)
    tx_b = np.concatenate([np.concatenate([t_b, t_b], axis=0),
                           np.concatenate([x_b1, x_b2], axis=0)], axis=1)
    u_b = np.zeros((2 * N_b, 1))

    # Collocation points
    tx_c = lb + (ub - lb) * lhs(2, N_c)

    # Convert to PyTorch tensors
    tx_i = torch.tensor(tx_i, dtype=torch.float32).to(device)
    u_i = torch.tensor(u_i, dtype=torch.float32).to(device)
    tx_b = torch.tensor(tx_b, dtype=torch.float32).to(device)
    u_b = torch.tensor(u_b, dtype=torch.float32).to(device)
    tx_c = torch.tensor(tx_c, dtype=torch.float32).to(device)

    return tx_c, tx_i, u_i, tx_b, u_b

tx_c, tx_i, u_i, tx_b, u_b = getData()

<div class="alert alert-danger" role="alert"> 
🔎 Setting Function of Plot

In [ ]:
def plotLoss(losses_dict, path, info=["I.C.+B.C.", "PDE"], colors=['blue', 'red']):
    fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(10, 6))
    axes[0].set_yscale("log")
    for i, j in zip(range(2), info):
        axes[i].plot(losses_dict[j.lower()], color=colors[i])
        axes[i].set_title(j)
        axes[i].set_xlabel("Iteration")
        axes[i].set_ylabel("Loss")
    plt.tight_layout()
    plt.savefig(path)
    plt.close()

def weights_init(m):
    if isinstance(m, torch.nn.Linear):
        torch.nn.init.xavier_normal_(m.weight.data)
        torch.nn.init.zeros_(m.bias.data)

<div class="alert alert-danger" role="alert"> 
🔎 Layer Class

In [ ]:
class Layer(torch.nn.Module):
    def __init__(self, n_in, n_out, activation):
        super().__init__()
        self.layer = torch.nn.Linear(n_in, n_out)
        self.activation = activation
    
    def forward(self, x):
        x = self.layer(x)
        if self.activation:
            x = self.activation(x)
        return x

<div class="alert alert-danger" role="alert"> 
🔎 DNN Class

In [ ]:
class DNN(torch.nn.Module):
    def __init__(self, dim_in=2, dim_out=1, n_layer=4, n_node=40, ub=ub, lb=lb, activation=torch.nn.Tanh()):
        super().__init__()
        self.net = torch.nn.ModuleList()
        self.net.append(Layer(dim_in, n_node, activation))
        for _ in range(n_layer):
            self.net.append(Layer(n_node, n_node, activation))
        self.net.append(Layer(n_node, dim_out, activation=None))
        self.ub = torch.tensor(ub, dtype=torch.float32).to(device)
        self.lb = torch.tensor(lb, dtype=torch.float32).to(device)
        self.net.apply(weights_init)
    
    def forward(self, x):
        x = (x - self.lb) / (self.ub - self.lb)  # Normalize input
        out = x
        for layer in self.net:
            out = layer(out)
        return out

<div class="alert alert-danger" role="alert"> 
🔎 PINN Class

In [ ]:
class PINN:
    def __init__(self):
        self.net = DNN(dim_in=2, dim_out=1, n_layer=4, n_node=40, ub=ub, lb=lb).to(device)
        self.lbfgs = torch.optim.LBFGS(
            self.net.parameters(),
            lr=1.0,
            max_iter=10000,
            max_eval=10000,
            tolerance_grad=1e-10,
            tolerance_change=np.finfo(float).eps,
            history_size=100,
            line_search_fn="strong_wolfe",
        )
        self.adam = torch.optim.Adam(self.net.parameters(), lr=0.001)
        self.losses = {"ic+bc": [], "pde": []}
        self.iter = 0
    
    def predict(self, tx):
        u = self.net(tx)
        return u
    
    def pde_loss(self, tx):
        tx = tx.clone()
        tx.requires_grad = True
        u = self.predict(tx)
        
        # Compute derivatives
        u_grad = torch.autograd.grad(u.sum(), tx, create_graph=True)[0]
        u_t = u_grad[:, 0:1]  # du/dt
        u_x = u_grad[:, 1:2]  # du/dx
        u_xx = torch.autograd.grad(u_x.sum(), tx, create_graph=True)[0][:, 1:2]  # d^2u/dx^2
        
        # PDE residual
        f = u_t + u * u_x - nu * u_xx
        mse_pde = torch.mean(torch.square(f))
        return mse_pde
    
    def ic_bc_loss(self, tx_i, u_i, tx_b, u_b):
        u_i_pred = self.predict(tx_i)
        u_b_pred = self.predict(tx_b)
        mse_ic = torch.mean(torch.square(u_i_pred - u_i))
        mse_bc = torch.mean(torch.square(u_b_pred - u_b))
        return mse_ic + mse_bc
    
    def closure(self):
        self.lbfgs.zero_grad()
        self.adam.zero_grad()
        mse_pde = self.pde_loss(tx_c)
        mse_ic_bc = self.ic_bc_loss(tx_i, u_i, tx_b, u_b)
        loss = mse_pde + mse_ic_bc
        loss.backward()
        
        self.losses["ic+bc"].append(mse_ic_bc.detach().cpu().item())
        self.losses["pde"].append(mse_pde.detach().cpu().item())
        self.iter += 1
        
        print(
            f"\r It: {self.iter} Loss: {loss.item():.5e} IC+BC: {mse_ic_bc.item():.3e} PDE: {mse_pde.item():.3e}",
            end=""
        )
        if self.iter % 100 == 0:
            print("")
        return loss

<div class="alert alert-danger" role="alert"> 
🔎 Train and evaluate the model

In [ ]:
if __name__ == "__main__":
    pinn = PINN()
    start_time = time.time()
    
    # Train with Adam
    for i in range(1000):
        pinn.closure()
        pinn.adam.step()
    
    # Train with L-BFGS
    pinn.lbfgs.step(pinn.closure)
    
    print(f"\n--- {time.time() - start_time:.2f} seconds ---")
    print(f"--- {(time.time() - start_time)/60:.2f} mins ---")
    
    # Save model
    torch.save(pinn.net.state_dict(), "burgers_pinn.pt")
    
    # Plot loss curves
    plotLoss(pinn.losses, "burgers_loss_curve.png", ["IC+BC", "PDE"])
    
    # Prediction points
    t = np.linspace(t_min, t_max, 100)
    x = np.linspace(x_min, x_max, 100)
    t_grid, x_grid = np.meshgrid(t, x)
    tx = np.stack([t_grid.flatten(), x_grid.flatten()], axis=-1)
    tx_tensor = torch.tensor(tx, dtype=torch.float32).to(device)
    
    # Predict solution
    with torch.no_grad():
        u = pinn.predict(tx_tensor).cpu().numpy().reshape(100, 100)
    
    # Plot solution
    fig, ax = plt.subplots(figsize=(8, 6))
    c = ax.pcolormesh(t_grid, x_grid, u, cmap='RdBu', vmin=-1, vmax=1)
    ax.set_xlabel('t')
    ax.set_ylabel('x')
    ax.set_title('PINN Solution: u(t, x)')
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="5%", pad=0.05)
    fig.colorbar(c, cax=cax, label='u(t, x)')
    
    # Plot cross-sections
    fig, axs = plt.subplots(1, 5, figsize=(15, 3), sharey=True)
    # t_sections = [0.0, 0.25, 0.5, 0.75, 1.0]
    t_sections = [0.0, 0.2, 0.45, 0.75, 1.0]
    for i, t_val in enumerate(t_sections):
        tx = np.stack([np.full_like(x, t_val), x], axis=-1)
        tx_tensor = torch.tensor(tx, dtype=torch.float32).to(device)
        with torch.no_grad():
            u = pinn.predict(tx_tensor).cpu().numpy()
        axs[i].plot(x, u, 'r-', label='PINN', color='k')
        if t_val == 0.0:
            axs[i].plot(x, -np.sin(np.pi * x), 'b--', label='Exact')
        axs[i].set_title(f't = {t_val}')
        axs[i].set_xlabel('x')
        if i == 0:
            axs[i].set_ylabel('u(t, x)')
            axs[i].legend()
    plt.tight_layout()
    plt.savefig("burgers_solution.png")
    plt.close()